# Modeling and Evaluation

In this section, we will build and evaluate our machine learning models. We will use the processed data from the previous steps and apply various modeling techniques to find the best-performing model for our task.


<a id="top"></a>
## Table of Contents

1. [Setup & Imports](#sec-1-imports)
2. [Load Dataset](#sec-2-load)
3. [Metrics & Utilities](#sec-3-utils)
4. [Model Baselines](#sec-4-baselines)
   - [4.1 Linear Regression](#sec-4-1-lin)
   - [4.2 Random Forest](#sec-4-2-rf)
5. [Model Comparison](#sec-5-compare)
6. [Submission File](#sec-6-submission)
7. [Kaggle Submission](#sec-7-kaggle)

[Back to top](#top)


<a id="sec-1-imports"></a>
## 1. Import Libraries


In [1]:
# --- Imports ---
import os
import pandas as pd
import numpy as np
import re
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor
from sklearn.feature_selection import RFE
from sklearn.base import clone


import os
import pandas as pd
import numpy as np
from datetime import datetime
from sklearn.metrics import mean_absolute_error, r2_score




<a id="sec-2-load"></a>
## 2. Load Dataset


In [2]:

# --- Define paths ---
data_dir = "../data/"
encoded_dir = os.path.join(data_dir, "feature_selection")

# --- Load encoded & scaled datasets ---
x_train = pd.read_csv(os.path.join(encoded_dir, "20_x_train.csv"))
y_train = pd.read_csv(os.path.join(encoded_dir, "20_y_train.csv")).squeeze()

x_val = pd.read_csv(os.path.join(encoded_dir, "20_x_val.csv"))
y_val = pd.read_csv(os.path.join(encoded_dir, "20_y_val.csv")).squeeze()

x_test = pd.read_csv(os.path.join(encoded_dir, "20_x_test.csv"))

# --- Sanity checks ---
print("Encoded datasets successfully loaded!")
print(f"x_train shape: {x_train.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"x_val shape:   {x_val.shape}")
print(f"y_val shape:   {y_val.shape}")
print(f"x_test shape:  {x_test.shape}")



Encoded datasets successfully loaded!
x_train shape: (55850, 21)
y_train shape: (55850,)
x_val shape:   (18617, 21)
y_val shape:   (18617,)
x_test shape:  (32567, 21)


<a id="sec-2-load"></a>
## 3. Metrics & Utilities


We use MAE because Kaggle for this course usually evaluates with MAE / it is robust to outliers in price.

In [3]:
# ======================================================
# Utilities: metrics & evaluation
# ======================================================

def evaluate_regression(y_true, y_pred):
    mae = float(mean_absolute_error(y_true, y_pred))
    r2 = float(r2_score(y_true, y_pred))
    return {"MAE": mae, "R2": r2}

def report_model(name, y_true, y_pred):
    m = evaluate_regression(y_true, y_pred)
    print(f"[{name}]  MAE: {m['MAE']:,.2f} | R²: {m['R2']:.4f}")
    return m


<a id="sec-2-load"></a>
## 4. Model Baselines


In [4]:
# ======================================================
# Baseline Model — Linear Regression
# ======================================================
from sklearn.linear_model import LinearRegression

lin = LinearRegression()
lin.fit(x_train, y_train)

y_val_pred_lin = lin.predict(x_val)
metrics_lin = report_model("LinearRegression (baseline)", y_val, y_val_pred_lin)


[LinearRegression (baseline)]  MAE: 2,941.40 | R²: 0.7797


In [5]:
# ======================================================
# Baseline Model - Ridge Regression
# ======================================================
from sklearn.linear_model import Ridge

ridge = Ridge(alpha=1.0)
ridge.fit(x_train, y_train)
y_val_pred_ridge = ridge.predict(x_val)
metrics_ridge = report_model("Ridge (baseline)", y_val, y_val_pred_ridge)


[Ridge (baseline)]  MAE: 2,941.41 | R²: 0.7797


In [6]:
# ======================================================
# Baseline Model - Lasso Regression
# ======================================================
from sklearn.linear_model import Lasso

lasso = Lasso(alpha=0.1)
lasso.fit(x_train, y_train)
y_val_pred_lasso = lasso.predict(x_val)
metrics_lasso = report_model("Lasso (baseline)", y_val, y_val_pred_lasso)

[Lasso (baseline)]  MAE: 2,941.37 | R²: 0.7797


/Users/karaca/src/MachineLearningProject-NOVAIMS2025/.venv/lib/python3.11/site-packages/sklearn/linear_model/_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 6.071e+10, tolerance: 5.284e+08
  model = cd_fast.enet_coordinate_descent(


In [7]:
# ======================================================
# Baseline Model - ElasticNet
# ======================================================

from sklearn.linear_model import ElasticNet


elasticnet = ElasticNet(alpha=0.1, l1_ratio=0.5)
elasticnet.fit(x_train, y_train)
y_val_pred_elasticnet = elasticnet.predict(x_val)
metrics_elasticnet = report_model("ElasticNet (baseline)", y_val, y_val_pred_elasticnet)


[ElasticNet (baseline)]  MAE: 3,019.47 | R²: 0.7635


In [8]:
# ======================================================
# Baseline Model — RandomForest
# ======================================================
from sklearn.ensemble import RandomForestRegressor

rf = RandomForestRegressor(
    n_estimators=400,
    max_depth=None,
    random_state=42,
    n_jobs=-1
)
rf.fit(x_train, y_train)

y_val_pred_rf = rf.predict(x_val)
metrics_rf = report_model("RandomForest (baseline)", y_val, y_val_pred_rf)


[RandomForest (baseline)]  MAE: 1,493.14 | R²: 0.9235


In [9]:
# ======================================================
# Baseline Model - DecisionTree
# ======================================================
from sklearn.tree import DecisionTreeRegressor

dt = DecisionTreeRegressor(
    max_depth=None,
    random_state=42,
)
dt.fit(x_train, y_train)
y_val_pred_dt = dt.predict(x_val)
metrics_dt = report_model("DecisionTree (baseline)", y_val, y_val_pred_dt)

[DecisionTree (baseline)]  MAE: 2,039.58 | R²: 0.8573


<a id="sec-4-comparison"></a>
## 4. Comparison of Models

In [11]:
# ======================================================
# Compare baselines & pick current best
# ======================================================
import pandas as pd

cmp = pd.DataFrame([
    {"model": "LinearRegression", **metrics_lin},
    {"model": "Ridge", **metrics_ridge},
    {"model": "Lasso", **metrics_lasso},
    {"model": "ElasticNet", **metrics_elasticnet},
    {"model": "RandomForest", **metrics_rf},
    {"model": "DecisionTree", **metrics_dt},
]).sort_values(by="MAE")

display(cmp)

best_name = cmp.iloc[0]["model"]
print(f"Current best (validation): {best_name}")


,model,MAE,R2
4,RandomForest,1493.139961,0.923462
5,DecisionTree,2039.576624,0.857274
2,Lasso,2941.371588,0.779743
0,LinearRegression,2941.402227,0.779742
1,Ridge,2941.413836,0.779739
3,ElasticNet,3019.472732,0.763483


Current best (validation): RandomForest


<a id="sec-5-hyperparameter-tuning"></a>
## 5. Hyperparameter Tuning & Selection

In [13]:
from sklearn.model_selection import ParameterGrid
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error

param_grid = {
    "n_estimators": [200, 400],
    "max_depth": [None, 20],
    "min_samples_split": [2, 5],
    "min_samples_leaf": [1, 2],
}

grid = list(ParameterGrid(param_grid))
n_total = len(grid)

best_score = float("inf")
best_params = None

for i, params in enumerate(grid, start=1):
    print(f"[{i}/{n_total}] Fitting with params: {params}")
    model = RandomForestRegressor(**params, random_state=42, n_jobs=-1)
    model.fit(x_train, y_train)
    y_pred = model.predict(x_val)
    mae = mean_absolute_error(y_val, y_pred)

    print(f"    -> MAE: {mae:.3f}")

    if mae < best_score:
        best_score = mae
        best_params = params
        print(f"    ✅ New best params: {best_params} (MAE={best_score:.3f})")

print("\nBest params:", best_params)
print("Best MAE:", best_score)


[1/16] Fitting with params: {'max_depth': None, 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 200}
    -> MAE: 1496.177
    ✅ New best params: {'max_depth': None, 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 200} (MAE=1496.177)
[2/16] Fitting with params: {'max_depth': None, 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 400}
    -> MAE: 1493.140
    ✅ New best params: {'max_depth': None, 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 400} (MAE=1493.140)
[3/16] Fitting with params: {'max_depth': None, 'min_samples_leaf': 1, 'min_samples_split': 5, 'n_estimators': 200}
    -> MAE: 1497.151
[4/16] Fitting with params: {'max_depth': None, 'min_samples_leaf': 1, 'min_samples_split': 5, 'n_estimators': 400}
    -> MAE: 1493.711
[5/16] Fitting with params: {'max_depth': None, 'min_samples_leaf': 2, 'min_samples_split': 2, 'n_estimators': 200}
    -> MAE: 1500.832
[6/16] Fitting with params: {'max_depth': None, 'min_samples_le

<a id="sec-2-load"></a>
## 6. Submission File to Kaggle

In [ ]:
# Build the best model 

# 2) Use the random forest model with the best hyperparameters found
best_model = RandomForestRegressor(**best_params, random_state=42, n_jobs=-1)
best_model.fit(pd.concat([x_train, x_val], axis=0), pd.concat([y_train, y_val], axis=0))


In [ ]:
# ======================================================
# Create Kaggle Submission (carID, price)
# ======================================================

# Load the raw test file to obtain carID
test_raw_path = os.path.join(data_dir, "test.csv")
test_raw = pd.read_csv(test_raw_path)

assert len(test_raw) == len(x_test), "Length mismatch between test.csv and x_test_final!"


y_test_pred = best_model.predict(x_test)

# (Optional) Round/clip according to competition rules
# Here we round to whole units, as in the sample, without allowing negative prices:
y_test_pred = np.clip(y_test_pred, a_min=0, a_max=None)
y_test_pred_rounded = np.rint(y_test_pred).astype(int)

print(test_raw["carID"].dtype)


# Build the submission DataFrame
submission = pd.DataFrame({
    "carID": test_raw["carID"],
    "price": y_test_pred_rounded  # optionally switch to y_test_pred if floats are allowed or preferred
})

# Save
sub_dir = os.path.join(data_dir, "submissions")
os.makedirs(sub_dir, exist_ok=True)
ts = datetime.now().strftime("%Y%m%d_%H%M")
sub_path = os.path.join(sub_dir, f"30_submission_{ts}.csv")
submission.to_csv(sub_path, index=False)

print(f"Submission saved to: {sub_path}")
display(submission.head(10))


Pipeline to the Kaggle Competition Submission

- you need to set up your api key in the .env file as KAGGLE_USERNAME and KAGGLE_KEY

In [ ]:
from dotenv import load_dotenv
from pathlib import Path
import os, json

env_path = Path("..") / ".env"   
load_dotenv(env_path, override=True)

kuser = os.getenv("KAGGLE_USERNAME")
kkey  = os.getenv("KAGGLE_KEY")

print("KAGGLE_USERNAME:", kuser)
print(".env loaded from:", env_path.resolve())


In [ ]:
!kaggle competitions submit -c cars4you -f {sub_path} -m "Group 21 submission"


In [ ]:
# Get the results of the submission
!kaggle competitions submissions -c cars4you